# Production Workflow Tutorial

**Purpose**: Learn how to train a model, validate it, and register it for production use  
**Author**: Hallett Lab  
**Date**: 2026-01-31

This tutorial covers the complete workflow from training to production:

1. **Train** a Tendril VAE model
2. **Validate** model quality with inference
3. **Register** the model as production-ready
4. **Use** registered models in applications
5. **Manage** the production registry

## Prerequisites

- Completed `tutorial_tendril_vae.ipynb` (recommended)
- Conda environment: `candescence_new`

```bash
conda activate candescence_new
```

## 1. Environment Setup

In [ ]:
import sys
from pathlib import Path

# Add src to path for development
src_path = Path("../src").resolve()
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
# Candescence imports
from candescence.core.config import TLVConfig
from candescence.core.production_registry import ProductionRegistry
from candescence.core.logging_config import get_logger, configure_logging
from candescence.tlv.factory import Factory
from candescence.tlv.inference import Inference

configure_logging(level='INFO')
logger = get_logger("tutorial_production")

print("All imports successful!")

## 2. Train a Model

First, we'll train a Tendril VAE model. For a detailed explanation of each parameter, see `tutorial_tendril_vae.ipynb`.

### 2.1 Configure the Experiment

The `TLVConfig` class manages where your model will be saved:

```
<refined>/{experiment_name}/{save_name}/
├── models/
│   ├── model.pth      <- Trained weights
│   └── args.json      <- Configuration
├── analyses/
├── loss_statistics/
└── meta_info/
```

In [ ]:
# Choose meaningful names for your experiment
EXPERIMENT_NAME = "production_tutorial"  # The project/experiment folder
SAVE_NAME = "model_v1"                   # This specific training run

# Create configuration
config = TLVConfig(
    experiment_name=EXPERIMENT_NAME,
    save_name=SAVE_NAME
)

# Create directory structure
config.create_directories()

print(f"Experiment: {EXPERIMENT_NAME}")
print(f"Run name: {SAVE_NAME}")
print(f"\nModel will be saved to:")
print(f"  {config.models_path}/model.pth")

In [ ]:
# Configure model architecture (Tendril VAE)
config.strategy = 14
config.architecture = 'tendril_vae'
config.latent_dim = 128
config.intermediate_dim = 256

# Conditioning
config.conditional_variables = ['average_hue']
config.conditional_decoder_fixed_values = {'average_hue': 0.5}

# Dataset
config.image_dimension = 128
config.restrict_to_day = 2
config.dataset_seed = 9954
config.training_seed = 9954

# For tutorial: small dataset, few epochs
# For production: increase these values
config.train_num = 400       # Production: 1200+
config.validation_num = 100  # Production: 400+
config.test_num = 100        # Production: 400+
config.number_epochs = 10    # Production: 100+
config.batch_size = 64

# Training parameters
config.vae_lr = 1e-4
config.kl_weight = 1
config.LPIPS_weight = 10
config.MSE_weight = 100

print("Configuration complete!")
print(f"Training samples: {config.train_num}")
print(f"Epochs: {config.number_epochs}")

### 2.2 Train the Model

In [ ]:
# Create factory and load data
factory = Factory(config)
factory.load_dataset()

print(f"Dataset loaded: {len(factory.dataset)} samples")

In [ ]:
# Prepare model and train
factory.prepare_vae()
factory.set_training_dataloader()

print("Starting training...")
factory.train_model()
print("\nTraining complete!")

In [ ]:
# Verify model was saved
model_path = config.models_path / "model.pth"
args_path = config.models_path / "args.json"

print(f"Model saved: {model_path.exists()}")
print(f"Args saved: {args_path.exists()}")

if model_path.exists():
    size_mb = model_path.stat().st_size / (1024 * 1024)
    print(f"Model size: {size_mb:.1f} MB")

## 3. Validate Model Quality

Before registering a model as production, validate its quality using the `Inference` class.

In [ ]:
# Run inference on test set
inference = Inference(factory)
inference.run_inference()

print("Inference complete!")

In [ ]:
# Check reconstruction quality
hsv_stats = inference.change_in_hsv()

print("Reconstruction Quality Metrics:")
print("=" * 40)
for key, value in hsv_stats.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")

In [ ]:
# Visualize sample reconstructions
from candescence.tlv.utilities import convert_rgb_transformed_2_rgb

# Get a few test samples
n_samples = 5
fig, axes = plt.subplots(2, n_samples, figsize=(3*n_samples, 6))

test_loader = factory.test_dataloader
batch = next(iter(test_loader))
img, cond_enc, cond_dec, _ = batch

img = img.to(config.device)
cond_enc = cond_enc.to(config.device).float()
cond_dec = cond_dec.to(config.device).float()

factory.vae.eval()
with torch.no_grad():
    z, mu, logvar, skip = factory.vae.encoder(img, cond_enc)
    reconstruction = factory.vae.decoder(z, skip, cond_dec)

for i in range(n_samples):
    # Original
    orig_img = convert_rgb_transformed_2_rgb(img[i].cpu())
    axes[0, i].imshow(orig_img)
    axes[0, i].set_title("Original")
    axes[0, i].axis('off')
    
    # Reconstruction
    rec_img = convert_rgb_transformed_2_rgb(reconstruction[i].cpu())
    axes[1, i].imshow(rec_img)
    axes[1, i].set_title("Reconstructed")
    axes[1, i].axis('off')

plt.suptitle("Model Quality Check: Original vs Reconstruction", fontsize=14)
plt.tight_layout()
plt.show()

### 3.1 Quality Checklist

Before promoting to production, verify:

- [ ] Reconstructions visually match originals
- [ ] Loss curves converged (check `loss_statistics/` folder)
- [ ] HSV changes are within acceptable range
- [ ] Model works on diverse samples (different morphologies)

## 4. Register as Production

Once satisfied with model quality, register it in the `ProductionRegistry`.

In [ ]:
# Initialize the production registry
registry = ProductionRegistry()

print(f"Registry location: {registry.registry_path}")
print(f"Current registered models: {registry.list_models()}")

In [ ]:
# Define model metadata
MODEL_ID = f"{EXPERIMENT_NAME}_{SAVE_NAME}"  # Unique identifier
MODEL_NAME = "Production Tutorial Model"     # Human-readable name
MODEL_VERSION = "1.0.0"                      # Semantic version
MODEL_DESCRIPTION = "Tutorial model trained on day 2 images with hue conditioning"

print(f"Model ID: {MODEL_ID}")
print(f"Model path: {model_path}")

In [ ]:
# Register the model
try:
    registry.register_model(
        model_id=MODEL_ID,
        model_path=model_path,
        name=MODEL_NAME,
        description=MODEL_DESCRIPTION,
        version=MODEL_VERSION,
        # These are auto-detected from args.json but can be overridden:
        # architecture="tendril_vae",
        # strategy=14,
        # latent_dim=128,
        # conditional_variables=['average_hue'],
    )
    print(f"Successfully registered model: {MODEL_ID}")
except ValueError as e:
    print(f"Model already registered: {e}")
    print("Use registry.update_model() to modify existing entries")

In [ ]:
# Verify registration
print("Registered models:")
for model_id in registry.list_models():
    info = registry.get_model_info(model_id)
    print(f"  - {model_id}")
    print(f"    Name: {info['name']}")
    print(f"    Version: {info['version']}")
    print(f"    Path: {info['model_path']}")
    print()

## 5. Use Registered Models

Now that the model is registered, it can be loaded from anywhere using the registry.

In [ ]:
# In any script or notebook, load a registered model:
from candescence.core.production_registry import ProductionRegistry
from candescence.interface.model_loader import load_tlv_model

# Get registry
registry = ProductionRegistry()

# List available models
print("Available production models:")
for choice in registry.get_model_choices():
    print(f"  - {choice['label']}")
    print(f"    ID: {choice['id']}")

In [ ]:
# Load a specific model by ID
model_info = registry.get_model_info(MODEL_ID)
model_path = registry.get_model_path(MODEL_ID)

print(f"Loading model: {model_info['name']}")
print(f"From: {model_path}")

# Load the model using the interface loader
args_path = model_path.parent / "args.json"
loaded_model = load_tlv_model(
    model_path=model_path,
    args_path=args_path
)

print(f"\nModel loaded successfully!")
print(f"Latent dim: {loaded_model.get_latent_dim()}")
print(f"Device: {loaded_model.device}")

## 6. Registry Management

The `ProductionRegistry` provides several management features.

In [ ]:
# Update model metadata (e.g., bump version after improvements)
# registry.update_model(
#     MODEL_ID,
#     version="1.0.1",
#     description="Updated description"
# )

# Unregister a model (does NOT delete files)
# registry.unregister_model(MODEL_ID)

# Export registry for backup
# registry.export_registry("/path/to/backup.json")

# Import registry from backup
# registry.import_registry("/path/to/backup.json", merge=True)

print("Registry management commands (uncomment to use)")

In [ ]:
# View the registry JSON file directly
import json

registry_file = registry.registry_path / "production_models.json"
if registry_file.exists():
    with open(registry_file) as f:
        data = json.load(f)
    print("Registry file contents:")
    print(json.dumps(data, indent=2, default=str))
else:
    print("No models registered yet")

## 7. Using in Streamlit Apps

Registered models can be used in the Latent Explorer app.

```bash
# Launch the app
streamlit run src/candescence/interface/apps/latent_explorer_app.py
```

In the app:
1. Select "From Model + Images" as the data source
2. Your registered models will appear in the dropdown
3. Select images to explore the latent space

## Summary

### The Production Workflow

| Step | Action | Code |
|------|--------|------|
| 1 | Configure experiment | `TLVConfig(experiment_name, save_name)` |
| 2 | Train model | `Factory.train_model()` |
| 3 | Validate quality | `Inference.run_inference()` |
| 4 | Register as production | `ProductionRegistry.register_model()` |
| 5 | Use anywhere | `registry.get_model_path(model_id)` |

### Key Files

| File | Purpose |
|------|---------|  
| `model.pth` | Trained model weights |
| `args.json` | Training configuration |
| `production_models.json` | Registry of production models |

### Best Practices

1. **Use meaningful names**: `experiment_name` should describe the project, `save_name` the specific run
2. **Version your models**: Use semantic versioning (1.0.0, 1.0.1, etc.)
3. **Validate before registering**: Always check reconstruction quality
4. **Document your models**: Use the `description` field to record training details
5. **Don't delete model files**: The registry only stores references; deleting files breaks the registry

In [ ]:
# Clean up (optional): Remove tutorial model from registry
# Uncomment to remove the tutorial model:

# registry.unregister_model(MODEL_ID)
# print(f"Removed {MODEL_ID} from registry")

print("Tutorial complete!")